In [0]:
CORRIDORS = {
    "I-70_STLKC": {
        "start": (-90.20, 38.63),
        "end":   (-94.58, 39.10),
        "label": "I-70 STL-KC",
        "n_segments": 26
    },
    "I-55_STLCHI": {
        "start": (-90.20, 38.63),
        "end":   (-87.63, 41.85),
        "label": "I-55 STL-CHI",
        "n_segments": 25
    },
    "I-64_STLCHI": {
        "start": (-90.20, 38.63),
        "end":   (-87.63, 41.85),
        "label": "I-64 STL-CHI",
        "n_segments": 14
    },
    "I-57_STLCHI": {
        "start": (-89.65, 37.73),
        "end":   (-87.63, 41.85),
        "label": "I-57 STL-CHI",
        "n_segments": 20
    },
    "I-80_STLCHI": {
        "start": (-88.50, 41.50),
        "end":   (-87.63, 41.85),
        "label": "I-80 CHI-END",
        "n_segments": 18
    },
}

print(f"Corridors defined: {len(CORRIDORS)}")

In [0]:
import random
random.seed(42)

def lerp(a, b, t):
    return a + (b - a) * t

road_segments = []
seg_id = 1

for corridor_id, cfg in CORRIDORS.items():
    n = cfg["n_segments"]
    lon1, lat1 = cfg["start"]
    lon2, lat2 = cfg["end"]
    
    for i in range(n):
        t_mid = (i + 0.5) / n
        mid_lon = lerp(lon1, lon2, t_mid)
        mid_lat = lerp(lat1, lat2, t_mid)
        
        road_segments.append({
            "segment_id": f"SEG_{seg_id:03d}",
            "corridor": corridor_id,
            "corridor_label": cfg["label"],
            "position_in_corridor": round(t_mid, 3),
            "midpoint_lon": round(mid_lon, 4),
            "midpoint_lat": round(mid_lat, 4),
            "length_miles": 10,
        })
        seg_id += 1

print(f"Generated {len(road_segments)} segments")

In [0]:
def urban_factor(t):
    near_endpoint = min(t, 1 - t)  # 0 at endpoints, 0.5 in middle
    return max(0.0, min(1.0, 1.0 - (near_endpoint * 1.6)))

# Test it
print(urban_factor(0.0))   # at STL - should be high
print(urban_factor(0.5))   # mid corridor - should be low
print(urban_factor(1.0))   # at KC - should be high

In [0]:
road_segments = []
seg_id = 1

for corridor_id, cfg in CORRIDORS.items():
    n = cfg["n_segments"]
    lon1, lat1 = cfg["start"]
    lon2, lat2 = cfg["end"]
    
    for i in range(n):
        t_mid = (i + 0.5) / n
        mid_lon = lerp(lon1, lon2, t_mid)
        mid_lat = lerp(lat1, lat2, t_mid)
        u = urban_factor(t_mid)
        
        straightness = round(random.gauss(85 - u * 20, 8), 1)
        straightness = max(40.0, min(100.0, straightness))
        
        road_segments.append({
            "segment_id": f"SEG_{seg_id:03d}",
            "corridor": corridor_id,
            "corridor_label": cfg["label"],
            "position_in_corridor": round(t_mid, 3),
            "midpoint_lon": round(mid_lon, 4),
            "midpoint_lat": round(mid_lat, 4),
            "length_miles": 10,
            "straightness_score": straightness,
        })
        seg_id += 1

print(f"Generated {len(road_segments)} segments")
print(road_segments[0])

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df_segments = spark.createDataFrame(road_segments)
df_segments.write.mode("overwrite").saveAsTable("road_segments")

print("Saved road_segments table")
spark.sql("SELECT COUNT(*) FROM road_segments").show()

In [0]:
traffic_counts = []

for seg in road_segments:
    t = seg["position_in_corridor"]
    u = urban_factor(t)
    
    aadt = int(random.gauss(35000 + u * 55000, 8000))
    aadt = max(8000, min(120000, aadt))
    
    ev_share = round(random.gauss(0.05 + u * 0.04, 0.015), 3)
    ev_share = max(0.02, min(0.12, ev_share))
    
    traffic_counts.append({
        "segment_id": seg["segment_id"],
        "aadt": aadt,
        "ev_share": ev_share,
        "ev_daily_vehicles": int(aadt * ev_share),
        "avg_speed_mph": round(random.gauss(72 - u * 10, 4), 1),
    })

df_traffic = spark.createDataFrame(traffic_counts)
df_traffic.write.mode("overwrite").saveAsTable("traffic_counts")
print("Saved traffic_counts")
spark.sql("SELECT COUNT(*) FROM traffic_counts").show()

In [0]:
import math

# WEATHER RISK
weather_risk = []
for seg in road_segments:
    lat = seg["midpoint_lat"]
    north_factor = max(0.0, min(1.0, (lat - 38.5) / 3.0))
    risk = round(random.gauss(30 + north_factor * 45, 8), 1)
    risk = max(5.0, min(95.0, risk))
    weather_risk.append({
        "segment_id": seg["segment_id"],
        "winter_risk_score": risk,
        "avg_snow_days_annual": round(random.gauss(10 + north_factor * 25, 5), 1),
    })

# INCIDENTS
incidents = []
for seg in road_segments:
    u = urban_factor(seg["position_in_corridor"])
    crash_rate = round(random.gauss(1.2 + u * 3.5, 0.6), 2)
    crash_rate = max(0.1, min(8.0, crash_rate))
    incidents.append({
        "segment_id": seg["segment_id"],
        "annual_crashes_per_mile": crash_rate,
    })

# INTERCHANGES
interchanges = []
ix_id = 1
for seg in road_segments:
    u = urban_factor(seg["position_in_corridor"])
    n_ix = max(0, int(random.gauss(2 + u * 6, 1.5)))
    for _ in range(n_ix):
        interchanges.append({
            "interchange_id": f"IX_{ix_id:04d}",
            "segment_id": seg["segment_id"],
            "interchange_type": random.choice(["full", "partial", "on_ramp_only"]),
        })
        ix_id += 1

# POWER INFRA
power_infra = []
for i in range(30):
    seg = random.choice(road_segments)
    power_infra.append({
        "infra_id": f"PWR_{i+1:03d}",
        "type": "substation",
        "capacity_mw": round(random.uniform(50, 500), 1),
        "lon": round(seg["midpoint_lon"] + random.gauss(0, 0.15), 4),
        "lat": round(seg["midpoint_lat"] + random.gauss(0, 0.15), 4),
    })

# ENV CONSTRAINTS
env_sites = [
    ("Carlyle Lake WMA",        -89.35, 38.60, "wetland"),
    ("Okaw Bottoms",            -88.50, 38.80, "wetland"),
    ("Shawnee NF",              -89.00, 37.50, "protected"),
    ("Kankakee River SRA",      -88.10, 41.15, "wetland"),
    ("Midewin National Tallgrass",-88.15,41.40,"protected"),
    ("Chain O Lakes",           -88.20, 42.48, "wetland"),
    ("Mingo NWR",               -90.30, 36.98, "wetland"),
    ("Mark Twain NWR",          -90.90, 39.65, "wetland"),
    ("Chautauqua NWR",          -90.10, 40.20, "wetland"),
    ("Hennepin IL Prairie",     -89.33, 41.25, "protected"),
    ("Banner Marsh",            -89.93, 40.52, "wetland"),
    ("Emiquon NWR",             -90.03, 40.48, "wetland"),
    ("Sangchris Lake SP",       -89.35, 39.68, "protected"),
    ("Lake Shelbyville",        -88.77, 39.40, "protected"),
    ("Crab Orchard NWR",        -89.05, 37.70, "protected"),
    ("Horseshoe Lake",          -89.35, 37.12, "wetland"),
    ("Prairie State Park MO",   -94.05, 37.85, "protected"),
]
env_constraints = []
for i, (name, lon, lat, etype) in enumerate(env_sites):
    env_constraints.append({
        "constraint_id": f"ENV_{i+1:02d}",
        "name": name,
        "type": etype,
        "centroid_lon": lon,
        "centroid_lat": lat,
    })

# SAVE ALL
spark.createDataFrame(weather_risk).write.mode("overwrite").saveAsTable("weather_risk")
spark.createDataFrame(incidents).write.mode("overwrite").saveAsTable("incidents")
spark.createDataFrame(interchanges).write.mode("overwrite").saveAsTable("interchanges")
spark.createDataFrame(power_infra).write.mode("overwrite").saveAsTable("power_infra")
spark.createDataFrame(env_constraints).write.mode("overwrite").saveAsTable("env_constraints")

print("All tables saved")

In [0]:
spark.sql("SHOW TABLES").show()